## Clinic Booking Agent — UI

This notebook launches the Gradio interface for the clinic booking agent.
It handles all user interaction including text input, voice input via Whisper,
and voice output via Edge TTS.

**What this file does:**
- Accepts typed messages and mic recordings from the user
- Transcribes voice input to text using Groq's Whisper Large V3
- Passes all messages to the agent's chat() function
- Converts the agent's text reply to speech using Edge TTS
- Renders the conversation in a Gradio Blocks chat interface

**Flow:**
User types or speaks → transcribe() → chat() → edge-tts speaks reply → Gradio displays it

In [ ]:
import nest_asyncio
nest_asyncio.apply()
import gradio as gr
from agent import chat
import edge_tts
import asyncio
import tempfile
import os
from dotenv import load_dotenv
from openai import OpenAI


In [ ]:
load_dotenv(override=True)

In [ ]:
groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")
groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
GROQMODEL = "openai/gpt-oss-20b"

In [ ]:
# Transcribes a recorded audio file to text using Groq's Whisper Large V3 model.
# Called when the user submits a mic recording instead of typing.
def transcribe(audio_path):
    with open(audio_path, "rb") as f:
        transcription = groq.audio.transcriptions.create(
            model="whisper-large-v3",
            file=f
        )
    return transcription.text

In [ ]:
# Converts the LLM's text reply to an MP3 file using Microsoft Edge TTS.
# Retries up to 3 times with backoff if the TTS service is unavailable.

async def safe_text_to_speech(text, retries=3):
    for attempt in range(retries):
        try:
            communicate = edge_tts.Communicate(text, voice="en-US-JennyNeural")
            with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as f:
                await communicate.save(f.name)
                return f.name
        except Exception as e:
            if attempt < retries - 1:
                await asyncio.sleep(1)  
            else:
                raise e

In [ ]:
# Handles typed text input. Calls chat() for the LLM reply,
# converts the reply to speech, and updates the chat history.
def respond(message, history):
    reply = chat(message, history)
    audio = asyncio.run(safe_text_to_speech(reply))
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": reply})
    return "", history, audio

In [ ]:
# Handles voice input. Transcribes the audio first,
# then passes the text to respond() for the same processing pipeline.
def respond_voice(audio_path, history):
    message = transcribe(audio_path)
    return respond(message, history)

In [ ]:
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(type="messages")
    audio_output = gr.Audio(autoplay=True, visible=True)
    
    
    with gr.Row():
        msg = gr.Textbox(placeholder="Type your message...", scale=4)
        mic = gr.Audio(sources=["microphone"], type="filepath", scale=1)
    
    msg.submit(respond, [msg, chatbot], [msg, chatbot, audio_output])
    mic.stop_recording(respond_voice, [mic, chatbot], [msg, chatbot, audio_output])

demo.launch(inbrowser=True)

In [ ]:
#cleared outputs